In [ ]:
#Notebook description

#This notebook is being used to evaluate the techinical market conditions of a single asset and assess
#the appropriate strategy to take in order to maximize returns.

In [ ]:
#take all compuation functions and put them in a separate file

#simplify the date x axis on the percent drawdown chart

#default the zoom range to a comfortable range, and create a dropdown to select the time range for Volatility section


#Remove the VIX charting, its redudnant now that we have the volatility models

In [ ]:
# Block 1: load core libraries and instantiate helpers and plotters
# Load libraries
import logging
import warnings
import json
import os
import numpy as np
from scipy.stats import skew, kurtosis
from scipy.interpolate import PchipInterpolator
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from plotly.subplots import make_subplots
from datetime import datetime
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from Quantapp.visualization import Plotter, LineChartPlotter, CandleStickPlotter, BarChartPlotter, add_sigma_reference_lines, add_mean_reference_line, add_std_annotations, add_zone_annotation, add_horizontal_zone_trace, build_time_range_buttons, build_detail_visibility_mask, build_visibility_mask
from Quantapp.analytics import TimeSeriesAnalytics as Rolling, Helper, Models, MomentumAnalytics, RiskDistributionAnalytics, RiskRelativeAnalytics, SeriesTransforms, calculate_zscore, calculate_max_drawdown, gini_coefficient, calculate_window_metrics
from Quantapp.data import MacroDataClient, normalize_benchmark_tickers, load_benchmark_data, align_series_to_common_index
warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")
logger.disabled = True
logger.propagate = False
rolling = Rolling()
series_transforms = SeriesTransforms()
momentum_analytics = MomentumAnalytics()
risk_relative_analytics = RiskRelativeAnalytics()
risk_distribution_analytics = RiskDistributionAnalytics()
qp = Plotter()
qe = MacroDataClient()
helper = Helper()
model = Models()
lineChartPlotter = LineChartPlotter()
candleStickPlotter = CandleStickPlotter()
barChartPlotter = BarChartPlotter()


In [ ]:
#PARAMETERS (Adjust parmeters here)
# Define parameters for the analysis (Adjust these as needed)
interval = '1d'
period     = '20y'
risk_free_rate = 0.02 / 252  # Annualized risk-free rate divided by trading days
#should be a string or None
ticker_str ='WMT'#ticker to analyze
benchmark_tickers = ['SPY', 'XLP']
length_of_plots = 20 #number of years of data to plot (after computing the period, this will be adjusted to the closest available data)
trading_strategy = 'position'  # Options: 'position', 'swing', or 'structural', # Determines the trading strategy to use for time frames


In [ ]:
# Block 1A: load treasury yields and build curve plots
fred_key = os.getenv("FRED_API_KEY")
if not fred_key and os.path.exists(".env"):
    with open(".env", "r", encoding="utf-8") as env_file:
        for line in env_file:
            if line.startswith("FRED_API_KEY="):
                fred_key = line.strip().split("=", 1)[1]
                break

if not fred_key:
    raise ValueError("FRED_API_KEY is required to load historical Treasury yields.")

treasury_client = MacroDataClient(fred_key=fred_key)
treasury_start_date = (pd.Timestamp.today().normalize() - pd.DateOffset(years=length_of_plots)).strftime("%Y-%m-%d")
treasury_yields = treasury_client.get_historical_treasury_yields(
    start_date=treasury_start_date,
).sort_index().dropna(how="all")

maturity_to_years = {
    "1M": 1 / 12,
    "3M": 3 / 12,
    "6M": 6 / 12,
    "1Y": 1,
    "2Y": 2,
    "3Y": 3,
    "5Y": 5,
    "7Y": 7,
    "10Y": 10,
    "20Y": 20,
    "30Y": 30,
}
bill_maturities = {"1M", "3M", "6M", "1Y"}
ordered_maturities = [maturity for maturity in maturity_to_years if maturity in treasury_yields.columns]
latest_curve_frame = treasury_yields[ordered_maturities].ffill().dropna(how="all")
asset_history = yf.Ticker(ticker_str).history(period="max", interval=interval)
asset_history = helper.simplify_datetime_index(asset_history)
asset_close = asset_history["Close"].dropna().sort_index()
curve_benchmark_tickers = normalize_benchmark_tickers(benchmark_tickers, ticker_str)
benchmark_close_map = {}
skipped_curve_benchmarks = []
for benchmark_symbol in curve_benchmark_tickers:
    benchmark_history = yf.Ticker(benchmark_symbol).history(period="max", interval=interval)
    benchmark_history = helper.simplify_datetime_index(benchmark_history)
    if benchmark_history.empty or "Close" not in benchmark_history:
        skipped_curve_benchmarks.append(benchmark_symbol)
        continue
    benchmark_close = benchmark_history["Close"].dropna().sort_index()
    if benchmark_close.empty:
        skipped_curve_benchmarks.append(benchmark_symbol)
        continue
    benchmark_close_map[benchmark_symbol] = benchmark_close
if latest_curve_frame.empty:
    raise ValueError("Treasury yield history is empty after loading FRED data.")
if asset_close.empty:
    raise ValueError(f"No price history available for {ticker_str}.")
comparison_end_candidates = [latest_curve_frame.index.max(), asset_close.index.max()]
comparison_end_candidates.extend(series.index.max() for series in benchmark_close_map.values())
comparison_end_date = min(comparison_end_candidates)
latest_curve_frame = latest_curve_frame.loc[:comparison_end_date]
asset_close = asset_close.loc[:comparison_end_date]
benchmark_close_map = {
    symbol: series.loc[:comparison_end_date]
    for symbol, series in benchmark_close_map.items()
}
if skipped_curve_benchmarks:
    print(f"Skipped yield-curve benchmark overlays with no data: {skipped_curve_benchmarks}")
latest_curve_date = latest_curve_frame.index[-1]
latest_curve = latest_curve_frame.iloc[-1].dropna()
curve_positions = list(range(len(latest_curve.index)))
curve_tick_text = latest_curve.index.tolist()
curve_year_values = np.array([maturity_to_years[maturity] for maturity in latest_curve.index], dtype=float)
curve_year_min = float(curve_year_values.min())
curve_year_max = float(curve_year_values.max())
bill_note_boundary = None
curve_region_end = len(curve_positions) - 0.5
if "1Y" in latest_curve.index and "2Y" in latest_curve.index:
    bill_note_boundary = latest_curve.index.get_loc("1Y") + 0.5
bill_region_mid = (bill_note_boundary - 0.5) / 2 if bill_note_boundary is not None else len(curve_positions) / 4
note_bond_region_mid = (
    (bill_note_boundary + curve_region_end) / 2 if bill_note_boundary is not None else len(curve_positions) * 0.75
)
bill_note_boundary_years = None
if "1Y" in latest_curve.index and "2Y" in latest_curve.index:
    bill_note_boundary_years = (maturity_to_years["1Y"] + maturity_to_years["2Y"]) / 2
bill_region_mid_years = (
    (curve_year_min + bill_note_boundary_years) / 2 if bill_note_boundary_years is not None else curve_year_min + ((curve_year_max - curve_year_min) * 0.25)
)
note_bond_region_mid_years = (
    (bill_note_boundary_years + curve_year_max) / 2 if bill_note_boundary_years is not None else curve_year_min + ((curve_year_max - curve_year_min) * 0.75)
)
maturity_date_offsets = {
    "1M": pd.DateOffset(months=1),
    "3M": pd.DateOffset(months=3),
    "6M": pd.DateOffset(months=6),
    "1Y": pd.DateOffset(years=1),
    "2Y": pd.DateOffset(years=2),
    "3Y": pd.DateOffset(years=3),
    "5Y": pd.DateOffset(years=5),
    "7Y": pd.DateOffset(years=7),
    "10Y": pd.DateOffset(years=10),
    "20Y": pd.DateOffset(years=20),
    "30Y": pd.DateOffset(years=30),
}

def deannualize_treasury_yield(yield_pct, maturity):
    years = maturity_to_years[maturity]
    yield_decimal = yield_pct / 100
    if maturity in bill_maturities:
        return (((1 + yield_decimal) ** years) - 1) * 100
    return (((1 + yield_decimal / 2) ** (2 * years)) - 1) * 100

def holding_period_asset_return(price_series, end_date, maturity):
    start_target = end_date - maturity_date_offsets[maturity]
    start_series = price_series.loc[:start_target]
    if start_series.empty:
        return np.nan
    start_price = start_series.iloc[-1]
    end_price = price_series.loc[:end_date].iloc[-1]
    if start_price <= 0 or end_price <= 0:
        return np.nan
    return ((end_price / start_price) - 1) * 100

def annualize_asset_return(price_series, end_date, maturity):
    years = maturity_to_years[maturity]
    holding_period_return = holding_period_asset_return(price_series, end_date, maturity)
    if pd.isna(holding_period_return):
        return np.nan
    gross_return = 1 + (holding_period_return / 100)
    if gross_return <= 0:
        return np.nan
    return ((gross_return ** (1 / years)) - 1) * 100

def build_interpolated_curve_data(curve_series):
    if curve_series is None or curve_series.empty or len(curve_series) < 2:
        return None
    x_values = np.array([maturity_to_years[maturity] for maturity in curve_series.index], dtype=float)
    y_values = curve_series.values.astype(float)
    sort_order = np.argsort(x_values)
    x_values = x_values[sort_order]
    y_values = y_values[sort_order]
    labels = [curve_series.index[idx] for idx in sort_order]
    x_plot_values = np.array([curve_tick_text.index(label) for label in labels], dtype=float)
    interpolator = PchipInterpolator(x_values, y_values)
    x_smooth = np.linspace(float(x_values.min()), float(x_values.max()), 300)
    y_smooth = interpolator(x_smooth)
    x_plot_smooth = np.interp(x_smooth, x_values, x_plot_values)
    return {
        "x_values": x_values,
        "x_plot_values": x_plot_values,
        "y_values": y_values,
        "x_smooth": x_smooth,
        "x_plot_smooth": x_plot_smooth,
        "y_smooth": y_smooth,
        "labels": labels,
    }

def build_historical_excess_return_series(price_series, treasury_frame, maturity):
    treasury_locked_yield_series = treasury_frame[maturity].ffill().dropna()
    if treasury_locked_yield_series.empty:
        return pd.Series(dtype=float), pd.Series(dtype=float)
    years = maturity_to_years[maturity]
    offset = maturity_date_offsets[maturity]
    candidate_dates = price_series.index.intersection(treasury_locked_yield_series.index)
    holding_period_excess_values = []
    annualized_excess_values = []
    valid_dates = []
    for end_date in candidate_dates:
        start_target = end_date - offset
        start_price_series = price_series.loc[:start_target]
        start_yield_series = treasury_locked_yield_series.loc[:start_target]
        if start_price_series.empty or start_yield_series.empty:
            continue
        start_price = start_price_series.iloc[-1]
        end_price = price_series.loc[:end_date].iloc[-1]
        locked_yield = start_yield_series.iloc[-1]
        if start_price <= 0 or end_price <= 0 or pd.isna(locked_yield):
            continue
        asset_holding_period_return = ((end_price / start_price) - 1) * 100
        treasury_holding_period_return = deannualize_treasury_yield(locked_yield, maturity)
        gross_asset_return = 1 + (asset_holding_period_return / 100)
        if gross_asset_return <= 0:
            asset_annualized_return = np.nan
        else:
            asset_annualized_return = ((gross_asset_return ** (1 / years)) - 1) * 100
        holding_period_excess_values.append(asset_holding_period_return - treasury_holding_period_return)
        annualized_excess_values.append(asset_annualized_return - locked_yield)
        valid_dates.append(end_date)
    return (
        pd.Series(holding_period_excess_values, index=valid_dates, dtype=float),
        pd.Series(annualized_excess_values, index=valid_dates, dtype=float),
    )

latest_hold_to_maturity_curve = pd.Series(
    {
        maturity: deannualize_treasury_yield(latest_curve[maturity], maturity)
        for maturity in latest_curve.index
    }
)
asset_annualized_curve = pd.Series(
    {
        maturity: annualize_asset_return(asset_close, latest_curve_date, maturity)
        for maturity in latest_curve.index
    }
).dropna()
asset_holding_period_curve = pd.Series(
    {
        maturity: holding_period_asset_return(asset_close, latest_curve_date, maturity)
        for maturity in latest_curve.index
    }
).dropna()
annualized_asset_curve_positions = [curve_tick_text.index(maturity) for maturity in asset_annualized_curve.index]
annualized_excess_curve = asset_annualized_curve - latest_curve.loc[asset_annualized_curve.index]
annualized_gap_x = []
annualized_gap_y = []
for maturity in asset_annualized_curve.index:
    position = curve_tick_text.index(maturity)
    annualized_gap_x.extend([position, position, None])
    annualized_gap_y.extend([latest_curve[maturity], asset_annualized_curve[maturity], None])
asset_curve_positions = [curve_tick_text.index(maturity) for maturity in asset_holding_period_curve.index]
holding_period_excess_curve = asset_holding_period_curve - latest_hold_to_maturity_curve.loc[asset_holding_period_curve.index]
holding_period_gap_x = []
holding_period_gap_y = []
for maturity in asset_holding_period_curve.index:
    position = curve_tick_text.index(maturity)
    holding_period_gap_x.extend([position, position, None])
    holding_period_gap_y.extend([latest_hold_to_maturity_curve[maturity], asset_holding_period_curve[maturity], None])
benchmark_curve_payloads = {}
benchmark_color_sequence = ["#16a34a", "#9333ea", "#d97706", "#dc2626", "#0891b2", "#7c3aed"]
for idx, (benchmark_symbol, benchmark_close) in enumerate(benchmark_close_map.items()):
    benchmark_annualized_curve = pd.Series(
        {
            maturity: annualize_asset_return(benchmark_close, latest_curve_date, maturity)
            for maturity in latest_curve.index
        }
    ).dropna()
    benchmark_holding_curve = pd.Series(
        {
            maturity: holding_period_asset_return(benchmark_close, latest_curve_date, maturity)
            for maturity in latest_curve.index
        }
    ).dropna()
    benchmark_curve_payloads[benchmark_symbol] = {
        "color": benchmark_color_sequence[idx % len(benchmark_color_sequence)],
        "annualized_curve": benchmark_annualized_curve,
        "annualized_positions": [curve_tick_text.index(maturity) for maturity in benchmark_annualized_curve.index],
        "annualized_excess_curve": benchmark_annualized_curve - latest_curve.loc[benchmark_annualized_curve.index],
        "holding_curve": benchmark_holding_curve,
        "holding_positions": [curve_tick_text.index(maturity) for maturity in benchmark_holding_curve.index],
        "holding_excess_curve": benchmark_holding_curve - latest_hold_to_maturity_curve.loc[benchmark_holding_curve.index],
    }
annualized_curve_interp = build_interpolated_curve_data(latest_curve)
deannualized_curve_interp = build_interpolated_curve_data(latest_hold_to_maturity_curve)
asset_annualized_curve_interp = build_interpolated_curve_data(asset_annualized_curve)
asset_holding_curve_interp = build_interpolated_curve_data(asset_holding_period_curve)
for benchmark_payload in benchmark_curve_payloads.values():
    benchmark_payload["annualized_interp"] = build_interpolated_curve_data(benchmark_payload["annualized_curve"])
    benchmark_payload["holding_interp"] = build_interpolated_curve_data(benchmark_payload["holding_curve"])
annualized_label_ceiling = latest_curve.max()
if not asset_annualized_curve.empty:
    annualized_label_ceiling = max(annualized_label_ceiling, asset_annualized_curve.max())
for benchmark_payload in benchmark_curve_payloads.values():
    if not benchmark_payload["annualized_curve"].empty:
        annualized_label_ceiling = max(annualized_label_ceiling, benchmark_payload["annualized_curve"].max())
annualized_label_y = annualized_label_ceiling * 0.96
deannualized_label_ceiling = latest_hold_to_maturity_curve.max()
if not asset_holding_period_curve.empty:
    deannualized_label_ceiling = max(deannualized_label_ceiling, asset_holding_period_curve.max())
for benchmark_payload in benchmark_curve_payloads.values():
    if not benchmark_payload["holding_curve"].empty:
        deannualized_label_ceiling = max(deannualized_label_ceiling, benchmark_payload["holding_curve"].max())
deannualized_label_y = deannualized_label_ceiling * 0.96
historical_excess_maturities = [
    maturity for maturity in ["3M", "1Y", "5Y", "10Y", "30Y"] if maturity in ordered_maturities
]
curve_overlay_label = f"{ticker_str} + Benchmarks" if benchmark_curve_payloads else ticker_str

treasury_fig = make_subplots(
    rows=3,
    cols=2,
    specs=[[{"colspan": 2}, None], [{}, {}], [{}, {}]],
    row_heights=[0.44, 0.28, 0.28],
    horizontal_spacing=0.08,
    vertical_spacing=0.1,
    subplot_titles=(
        "Historical Treasury Yields",
        f"Annualized Yield Curve vs {curve_overlay_label} ({latest_curve_date:%Y-%m-%d})",
        f"De-Annualized Yield Curve vs {curve_overlay_label} ({latest_curve_date:%Y-%m-%d})",
        f"Interpolated Annualized Curves ({latest_curve_date:%Y-%m-%d})",
        f"Interpolated De-Annualized Curves ({latest_curve_date:%Y-%m-%d})",
    ),
)

for maturity in ordered_maturities:
    series = treasury_yields[maturity].dropna()
    if series.empty:
        continue
    treasury_fig.add_trace(
        go.Scatter(
            x=series.index,
            y=series,
            mode="lines",
            name=maturity,
            line=dict(width=1.6),
        ),
        row=1,
        col=1,
    )

treasury_fig.add_trace(
    go.Scatter(
        x=curve_positions,
        y=latest_curve.values.tolist(),
        mode="lines+markers",
        name="Latest Yield Curve",
        line=dict(color="#0f172a", width=3),
        marker=dict(size=9, color="#0f766e"),
        showlegend=False,
    ),
    row=2,
    col=1,
)

if annualized_gap_x:
    treasury_fig.add_trace(
        go.Scatter(
            x=annualized_gap_x,
            y=annualized_gap_y,
            mode="lines",
            name="Annualized Excess Return Gap",
            line=dict(color="rgba(37, 99, 235, 0.45)", width=2, dash="dot"),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=2,
        col=1,
    )

if not asset_annualized_curve.empty:
    treasury_fig.add_trace(
        go.Scatter(
            x=annualized_asset_curve_positions,
            y=asset_annualized_curve.values.tolist(),
            mode="lines+markers",
            name=f"{ticker_str} Annualized Return",
            line=dict(color="#2563eb", width=3),
            marker=dict(size=9, color="#1d4ed8", symbol="diamond"),
            customdata=np.column_stack((asset_annualized_curve.index, annualized_excess_curve.values)),
            hovertemplate=(
                "Maturity=%{customdata[0]}<br>"
                + f"{ticker_str} annualized return=%{{y:.2f}}%<br>"
                + "Excess vs Treasury=%{customdata[1]:.2f}%<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

for benchmark_symbol, benchmark_payload in benchmark_curve_payloads.items():
    if benchmark_payload["annualized_curve"].empty:
        continue
    treasury_fig.add_trace(
        go.Scatter(
            x=benchmark_payload["annualized_positions"],
            y=benchmark_payload["annualized_curve"].values.tolist(),
            mode="lines+markers",
            name=f"{benchmark_symbol} Annualized Return",
            legendgroup=benchmark_symbol,
            line=dict(color=benchmark_payload["color"], width=2.4, dash="dash"),
            marker=dict(size=8, color=benchmark_payload["color"], symbol="circle-open"),
            customdata=np.column_stack((benchmark_payload["annualized_curve"].index, benchmark_payload["annualized_excess_curve"].values)),
            hovertemplate=(
                "Maturity=%{customdata[0]}<br>"
                + f"{benchmark_symbol} annualized return=%{{y:.2f}}%<br>"
                + "Excess vs Treasury=%{customdata[1]:.2f}%<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

treasury_fig.add_trace(
    go.Scatter(
        x=curve_positions,
        y=latest_hold_to_maturity_curve.values.tolist(),
        mode="lines+markers",
        name="De-Annualized Curve",
        line=dict(color="#7c2d12", width=3),
        marker=dict(size=9, color="#ea580c"),
        showlegend=False,
    ),
    row=2,
    col=2,
)

for benchmark_symbol, benchmark_payload in benchmark_curve_payloads.items():
    if benchmark_payload["holding_curve"].empty:
        continue
    treasury_fig.add_trace(
        go.Scatter(
            x=benchmark_payload["holding_positions"],
            y=benchmark_payload["holding_curve"].values.tolist(),
            mode="lines+markers",
            name=f"{benchmark_symbol} Holding-Period Return",
            legendgroup=benchmark_symbol,
            line=dict(color=benchmark_payload["color"], width=2.4, dash="dash"),
            marker=dict(size=8, color=benchmark_payload["color"], symbol="circle-open"),
            customdata=np.column_stack((benchmark_payload["holding_curve"].index, benchmark_payload["holding_excess_curve"].values)),
            hovertemplate=(
                "Maturity=%{customdata[0]}<br>"
                + f"{benchmark_symbol} holding-period return=%{{y:.2f}}%<br>"
                + "Excess vs Treasury=%{customdata[1]:.2f}%<extra></extra>"
            ),
        ),
        row=2,
        col=2,
    )

if holding_period_gap_x:
    treasury_fig.add_trace(
        go.Scatter(
            x=holding_period_gap_x,
            y=holding_period_gap_y,
            mode="lines",
            name="Holding-Period Excess Return Gap",
            line=dict(color="rgba(37, 99, 235, 0.45)", width=2, dash="dot"),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=2,
        col=2,
    )

if not asset_holding_period_curve.empty:
    treasury_fig.add_trace(
        go.Scatter(
            x=asset_curve_positions,
            y=asset_holding_period_curve.values.tolist(),
            mode="lines+markers",
            name=f"{ticker_str} Holding-Period Return",
            line=dict(color="#2563eb", width=3),
            marker=dict(size=9, color="#1d4ed8", symbol="diamond"),
            customdata=np.column_stack((asset_holding_period_curve.index, holding_period_excess_curve.values)),
            hovertemplate=(
                "Maturity=%{customdata[0]}<br>"
                + f"{ticker_str} holding-period return=%{{y:.2f}}%<br>"
                + "Excess vs Treasury=%{customdata[1]:.2f}%<extra></extra>"
            ),
        ),
        row=2,
        col=2,
    )

if annualized_curve_interp is not None:
    treasury_fig.add_trace(
        go.Scatter(
            x=annualized_curve_interp["x_plot_smooth"],
            y=annualized_curve_interp["y_smooth"],
            mode="lines",
            name="Interpolated Treasury Yield Curve",
            line=dict(color="#0f172a", width=3),
            showlegend=False,
        ),
        row=3,
        col=1,
    )
    treasury_fig.add_trace(
        go.Scatter(
            x=annualized_curve_interp["x_plot_values"],
            y=annualized_curve_interp["y_values"],
            mode="markers",
            marker=dict(size=7, color="#0f766e"),
            text=annualized_curve_interp["labels"],
            hovertemplate="Maturity=%{text}<br>Treasury yield=%{y:.2f}%<extra></extra>",
            showlegend=False,
        ),
        row=3,
        col=1,
    )

if asset_annualized_curve_interp is not None:
    treasury_fig.add_trace(
        go.Scatter(
            x=asset_annualized_curve_interp["x_plot_smooth"],
            y=asset_annualized_curve_interp["y_smooth"],
            mode="lines",
            line=dict(color="#2563eb", width=3),
            showlegend=False,
        ),
        row=3,
        col=1,
    )
    treasury_fig.add_trace(
        go.Scatter(
            x=asset_annualized_curve_interp["x_plot_values"],
            y=asset_annualized_curve_interp["y_values"],
            mode="markers",
            marker=dict(size=7, color="#1d4ed8", symbol="diamond"),
            text=asset_annualized_curve_interp["labels"],
            hovertemplate=(
                "Maturity=%{text}<br>"
                + f"{ticker_str} annualized return=%{{y:.2f}}%<extra></extra>"
            ),
            showlegend=False,
        ),
        row=3,
        col=1,
    )

for benchmark_symbol, benchmark_payload in benchmark_curve_payloads.items():
    benchmark_interp = benchmark_payload.get("annualized_interp")
    if benchmark_interp is None:
        continue
    treasury_fig.add_trace(
        go.Scatter(
            x=benchmark_interp["x_plot_smooth"],
            y=benchmark_interp["y_smooth"],
            mode="lines",
            line=dict(color=benchmark_payload["color"], width=2.2, dash="dash"),
            showlegend=False,
        ),
        row=3,
        col=1,
    )
    treasury_fig.add_trace(
        go.Scatter(
            x=benchmark_interp["x_plot_values"],
            y=benchmark_interp["y_values"],
            mode="markers",
            marker=dict(size=6, color=benchmark_payload["color"], symbol="circle-open"),
            text=benchmark_interp["labels"],
            hovertemplate=(
                "Maturity=%{text}<br>"
                + f"{benchmark_symbol} annualized return=%{{y:.2f}}%<extra></extra>"
            ),
            showlegend=False,
        ),
        row=3,
        col=1,
    )

if deannualized_curve_interp is not None:
    treasury_fig.add_trace(
        go.Scatter(
            x=deannualized_curve_interp["x_plot_smooth"],
            y=deannualized_curve_interp["y_smooth"],
            mode="lines",
            name="Interpolated De-Annualized Curve",
            line=dict(color="#7c2d12", width=3),
            showlegend=False,
        ),
        row=3,
        col=2,
    )
    treasury_fig.add_trace(
        go.Scatter(
            x=deannualized_curve_interp["x_plot_values"],
            y=deannualized_curve_interp["y_values"],
            mode="markers",
            marker=dict(size=7, color="#ea580c"),
            text=deannualized_curve_interp["labels"],
            hovertemplate="Maturity=%{text}<br>Treasury holding-period return=%{y:.2f}%<extra></extra>",
            showlegend=False,
        ),
        row=3,
        col=2,
    )

if asset_holding_curve_interp is not None:
    treasury_fig.add_trace(
        go.Scatter(
            x=asset_holding_curve_interp["x_plot_smooth"],
            y=asset_holding_curve_interp["y_smooth"],
            mode="lines",
            line=dict(color="#2563eb", width=3),
            showlegend=False,
        ),
        row=3,
        col=2,
    )
    treasury_fig.add_trace(
        go.Scatter(
            x=asset_holding_curve_interp["x_plot_values"],
            y=asset_holding_curve_interp["y_values"],
            mode="markers",
            marker=dict(size=7, color="#1d4ed8", symbol="diamond"),
            text=asset_holding_curve_interp["labels"],
            hovertemplate=(
                "Maturity=%{text}<br>"
                + f"{ticker_str} holding-period return=%{{y:.2f}}%<extra></extra>"
            ),
            showlegend=False,
        ),
        row=3,
        col=2,
    )

for benchmark_symbol, benchmark_payload in benchmark_curve_payloads.items():
    benchmark_interp = benchmark_payload.get("holding_interp")
    if benchmark_interp is None:
        continue
    treasury_fig.add_trace(
        go.Scatter(
            x=benchmark_interp["x_plot_smooth"],
            y=benchmark_interp["y_smooth"],
            mode="lines",
            line=dict(color=benchmark_payload["color"], width=2.2, dash="dash"),
            showlegend=False,
        ),
        row=3,
        col=2,
    )
    treasury_fig.add_trace(
        go.Scatter(
            x=benchmark_interp["x_plot_values"],
            y=benchmark_interp["y_values"],
            mode="markers",
            marker=dict(size=6, color=benchmark_payload["color"], symbol="circle-open"),
            text=benchmark_interp["labels"],
            hovertemplate=(
                "Maturity=%{text}<br>"
                + f"{benchmark_symbol} holding-period return=%{{y:.2f}}%<extra></extra>"
            ),
            showlegend=False,
        ),
        row=3,
        col=2,
    )

if bill_note_boundary is not None:
    treasury_fig.add_vrect(
        x0=bill_note_boundary,
        x1=curve_region_end,
        fillcolor="rgba(148, 163, 184, 0.18)",
        line_width=0,
        row=2,
        col=1,
    )
    treasury_fig.add_vrect(
        x0=bill_note_boundary,
        x1=curve_region_end,
        fillcolor="rgba(148, 163, 184, 0.18)",
        line_width=0,
        row=2,
        col=2,
    )
    treasury_fig.add_annotation(
        x=bill_region_mid,
        y=annualized_label_y,
        text="Bills",
        showarrow=False,
        font=dict(size=11, color="#334155"),
        bgcolor="rgba(255, 255, 255, 0.85)",
        bordercolor="rgba(148, 163, 184, 0.5)",
        row=2,
        col=1,
    )
    treasury_fig.add_annotation(
        x=note_bond_region_mid,
        y=annualized_label_y,
        text="Notes/Bonds",
        showarrow=False,
        font=dict(size=11, color="#334155"),
        bgcolor="rgba(255, 255, 255, 0.85)",
        bordercolor="rgba(148, 163, 184, 0.5)",
        row=2,
        col=1,
    )
    treasury_fig.add_annotation(
        x=bill_region_mid,
        y=deannualized_label_y,
        text="Bills",
        showarrow=False,
        font=dict(size=11, color="#334155"),
        bgcolor="rgba(255, 255, 255, 0.85)",
        bordercolor="rgba(148, 163, 184, 0.5)",
        row=2,
        col=2,
    )
    treasury_fig.add_annotation(
        x=note_bond_region_mid,
        y=deannualized_label_y,
        text="Notes/Bonds",
        showarrow=False,
        font=dict(size=11, color="#334155"),
        bgcolor="rgba(255, 255, 255, 0.85)",
        bordercolor="rgba(148, 163, 184, 0.5)",
        row=2,
        col=2,
    )
    treasury_fig.add_vrect(
        x0=bill_note_boundary,
        x1=curve_region_end,
        fillcolor="rgba(148, 163, 184, 0.18)",
        line_width=0,
        row=3,
        col=1,
    )
    treasury_fig.add_vrect(
        x0=bill_note_boundary,
        x1=curve_region_end,
        fillcolor="rgba(148, 163, 184, 0.18)",
        line_width=0,
        row=3,
        col=2,
    )
    treasury_fig.add_annotation(
        x=bill_region_mid,
        y=annualized_label_y,
        text="Bills",
        showarrow=False,
        font=dict(size=11, color="#334155"),
        bgcolor="rgba(255, 255, 255, 0.85)",
        bordercolor="rgba(148, 163, 184, 0.5)",
        row=3,
        col=1,
    )
    treasury_fig.add_annotation(
        x=note_bond_region_mid,
        y=annualized_label_y,
        text="Notes/Bonds",
        showarrow=False,
        font=dict(size=11, color="#334155"),
        bgcolor="rgba(255, 255, 255, 0.85)",
        bordercolor="rgba(148, 163, 184, 0.5)",
        row=3,
        col=1,
    )
    treasury_fig.add_annotation(
        x=bill_region_mid,
        y=deannualized_label_y,
        text="Bills",
        showarrow=False,
        font=dict(size=11, color="#334155"),
        bgcolor="rgba(255, 255, 255, 0.85)",
        bordercolor="rgba(148, 163, 184, 0.5)",
        row=3,
        col=2,
    )
    treasury_fig.add_annotation(
        x=note_bond_region_mid,
        y=deannualized_label_y,
        text="Notes/Bonds",
        showarrow=False,
        font=dict(size=11, color="#334155"),
        bgcolor="rgba(255, 255, 255, 0.85)",
        bordercolor="rgba(148, 163, 184, 0.5)",
        row=3,
        col=2,
    )

treasury_fig.update_xaxes(title_text="Date", row=1, col=1)
treasury_fig.update_yaxes(title_text="Yield (%)", row=1, col=1)
treasury_fig.update_xaxes(
    title_text="Maturity",
    tickmode="array",
    tickvals=curve_positions,
    ticktext=curve_tick_text,
    range=[-0.5, curve_region_end],
    row=2,
    col=1,
)
treasury_fig.update_yaxes(title_text="Annualized Return / Yield (%)", row=2, col=1)
treasury_fig.update_xaxes(
    title_text="Maturity",
    tickmode="array",
    tickvals=curve_positions,
    ticktext=curve_tick_text,
    range=[-0.5, curve_region_end],
    row=2,
    col=2,
)
treasury_fig.update_yaxes(title_text="Holding-Period Return (%)", row=2, col=2)
treasury_fig.update_xaxes(
    title_text="Maturity",
    tickmode="array",
    tickvals=curve_positions,
    ticktext=curve_tick_text,
    range=[-0.5, curve_region_end],
    row=3,
    col=1,
)
treasury_fig.update_yaxes(title_text="Annualized Return / Yield (%)", row=3, col=1)
treasury_fig.update_xaxes(
    title_text="Maturity",
    tickmode="array",
    tickvals=curve_positions,
    ticktext=curve_tick_text,
    range=[-0.5, curve_region_end],
    row=3,
    col=2,
)
treasury_fig.update_yaxes(title_text="Holding-Period Return (%)", row=3, col=2)
treasury_fig.update_layout(
    title=f"US Treasury Yield History with {curve_overlay_label} Matched-Horizon Return Overlays",
    template="plotly_white",
    height=1450,
    legend_title_text="Series",
    hovermode="x unified",
)
treasury_fig.show()

historical_holding_excess_map = {}
historical_annualized_excess_map = {}
historical_color_sequence = px.colors.qualitative.Bold
for maturity in historical_excess_maturities:
    holding_excess_series, annualized_excess_series = build_historical_excess_return_series(
        price_series=asset_close,
        treasury_frame=latest_curve_frame,
        maturity=maturity,
    )
    if not holding_excess_series.empty:
        historical_holding_excess_map[maturity] = holding_excess_series
    if not annualized_excess_series.empty:
        historical_annualized_excess_map[maturity] = annualized_excess_series

if historical_holding_excess_map or historical_annualized_excess_map:
    historical_excess_fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=(
            f"{ticker_str} Historical Holding-Period Excess Return vs Locked Treasury",
            f"{ticker_str} Historical Annualized Excess Return vs Locked Treasury",
        ),
    )
    for idx, maturity in enumerate(historical_excess_maturities):
        color = historical_color_sequence[idx % len(historical_color_sequence)]
        holding_series = historical_holding_excess_map.get(maturity)
        annualized_series = historical_annualized_excess_map.get(maturity)
        if holding_series is not None and not holding_series.empty:
            historical_excess_fig.add_trace(
                go.Scatter(
                    x=holding_series.index,
                    y=holding_series.values,
                    mode="lines",
                    name=maturity,
                    legendgroup=maturity,
                    line=dict(color=color, width=2.2),
                ),
                row=1,
                col=1,
            )
        if annualized_series is not None and not annualized_series.empty:
            historical_excess_fig.add_trace(
                go.Scatter(
                    x=annualized_series.index,
                    y=annualized_series.values,
                    mode="lines",
                    name=maturity,
                    legendgroup=maturity,
                    line=dict(color=color, width=2.2, dash="dot"),
                    showlegend=False,
                ),
                row=2,
                col=1,
            )
    historical_excess_fig.add_hline(y=0, line_color="#475569", line_width=1, line_dash="dot", row=1, col=1)
    historical_excess_fig.add_hline(y=0, line_color="#475569", line_width=1, line_dash="dot", row=2, col=1)
    historical_excess_fig.update_yaxes(title_text="Excess Return (%)", row=1, col=1)
    historical_excess_fig.update_yaxes(title_text="Excess Return (%)", row=2, col=1)
    historical_excess_fig.update_xaxes(title_text="Date", row=2, col=1)
    historical_excess_fig.update_layout(
        title=f"{ticker_str} Historical Excess Returns vs Locked Treasury Benchmarks",
        template="plotly_white",
        height=900,
        legend_title_text="Maturity",
        hovermode="x unified",
    )
    historical_excess_fig.show()

# Compare the last 3 months of the active asset's price performance to the 3-month Treasury return
# that could have been locked in at the start of the same 3-month window.
asset_vs_bond_3m = pd.concat(
    [
        asset_close.rename("Asset_Close"),
        treasury_yields[["3M"]],
    ],
    axis=1,
    join="inner",
).dropna()
asset_vs_bond_3m["asset_trailing_3m_return"] = asset_vs_bond_3m["Asset_Close"].pct_change(63)
asset_vs_bond_3m["bond_locked_3m_return"] = (
    (1 + asset_vs_bond_3m["3M"].shift(63) / 100) ** (3 / 12) - 1
)
asset_vs_bond_3m["spread"] = (
    asset_vs_bond_3m["asset_trailing_3m_return"] - asset_vs_bond_3m["bond_locked_3m_return"]
)
asset_vs_bond_3m = asset_vs_bond_3m.dropna()

latest_asset_vs_bond = asset_vs_bond_3m.iloc[-1]
latest_asset_vs_bond_date = asset_vs_bond_3m.index[-1]
beat_bond_text = "Yes" if latest_asset_vs_bond["spread"] > 0 else "No"
print(
    f"As of {latest_asset_vs_bond_date:%Y-%m-%d}: {ticker_str} returned {latest_asset_vs_bond['asset_trailing_3m_return']:.2%} "
    f"over the last 3 months versus {latest_asset_vs_bond['bond_locked_3m_return']:.2%} from holding the "
    f"3-month Treasury bought 3 months earlier. Did {ticker_str} beat the bond? {beat_bond_text}."
)



In [ ]:
# Block 2: organize structural, position, swing timeframe lists

# Define strategy timeframe profiles
TIMEFRAME_PROFILES = {
    'swing': {'short': 3, 'mid': 9, 'long': 21},
    'position': {'short': 21, 'mid': 50, 'long': 200},
    'structural': {'short': 200, 'mid': 500, 'long': 1000},
}

strategy = str(trading_strategy).strip().lower()
if strategy not in TIMEFRAME_PROFILES:
    raise ValueError(
        f"Invalid trading_strategy '{trading_strategy}'. "
        f"Expected one of: {list(TIMEFRAME_PROFILES.keys())}"
    )

time_frame_map = TIMEFRAME_PROFILES[strategy]
time_frame_short = time_frame_map['short']
time_frame_mid = time_frame_map['mid']
time_frame_long = time_frame_map['long']

return_frequencies = ('monthly', 'weekly', 'daily')


In [ ]:
# Block 3: load and align asset, benchmark, and return data

# Download and normalize asset-level data
ticker = yf.Ticker(ticker_str).history(period=period, interval=interval)
vix = yf.Ticker('^VIX').history(period=period, interval=interval)
ticker = helper.simplify_datetime_index(ticker)
vix = helper.simplify_datetime_index(vix)

# Download benchmark data once and keep it in collections for downstream cells
benchmark_tickers = normalize_benchmark_tickers(benchmark_tickers, ticker_str)
benchmark_data, skipped_benchmarks = load_benchmark_data(benchmark_tickers, period, interval, helper)
if skipped_benchmarks:
    print(f'Skipped benchmarks with no data: {skipped_benchmarks}')

analysis_index, ticker, vix, benchmark_data = align_series_to_common_index(ticker, vix, benchmark_data)

# Calculate asset and benchmark returns for the frequencies used elsewhere in the notebook
ticker_returns = {frequency: series_transforms.returns(ticker, frequency=frequency) for frequency in return_frequencies}
ticker_monthly_returns = ticker_returns['monthly']
ticker_weekly_returns = ticker_returns['weekly']
ticker_daily_returns = ticker_returns['daily']

benchmark_returns = {
    symbol: {frequency: series_transforms.returns(frame, frequency=frequency) for frequency in return_frequencies}
    for symbol, frame in benchmark_data.items()
}

vix_returns = {frequency: series_transforms.returns(vix, frequency=frequency) for frequency in return_frequencies}
vix_monthly_returns = vix_returns['monthly']
vix_weekly_returns = vix_returns['weekly']
vix_daily_returns = vix_returns['daily']


In [ ]:
# Block 4: display regime-change candlestick with Bollinger bands

#REGIME CHANGES: Candlestick with RSI and Bollinger Bands
candleStickPlotter.create_candlestick_fig(ticker, period='2y', bollinger_window=50, title="Candlestick with 50-Period Bollinger Bands")

In [ ]:
# Block 5: plot percentage drop from historical peaks

#REGIME CHANGES: Percentage Drop from Highest Peak
n = int(252 / 2)
qp.plot_percentage_drop(ticker, n=n, title=f'{ticker_str} Percentage Drop from Highest Peak')

In [ ]:
# Block 6: compute rolling Sharpe windows, momentum histograms, and volatility

window_sizes = list(range(3, 201))

momentum_diagnostics_context = momentum_analytics.build_momentum_window_diagnostics_context(
    close_series=ticker['Close'],
    window_sizes=window_sizes,
    highlight_windows=(7, 21, 50, 200),
    surface_years=10,
)

momentum_diagnostic_figs = lineChartPlotter.plot_momentum_window_diagnostics(
    diagnostics_context=momentum_diagnostics_context,
    ticker_label=ticker_str,
)

momentum_diagnostic_figs['optimal_window'].show()
momentum_diagnostic_figs['optimal_window_histogram'].show()
momentum_diagnostic_figs['sharpe_mean_median'].show()
momentum_diagnostic_figs['volatility_mean_median'].show()
momentum_diagnostic_figs['sharpe_surface_3d'].show()


In [ ]:
# Block 7: build VIX Fix series and overlay standard deviation bands

#Volatility: VIX FIX 

ticker_vix_fix = rolling.vix_fix(ticker)
benchmark_vix_fix = {symbol: rolling.vix_fix(frame) for symbol, frame in benchmark_data.items()}

qp.plot_series_with_stdev_bands(
    ticker_vix_fix,
    title='VIX Fix with Mean and Standard Deviations',
    stdev_values=[-0.5, 0.5, 1.5, 3]
)


In [ ]:
# Block 8: visualize monthly, weekly, and daily seasonality patterns

#Seasonality: Monthly, Weekly, and Daily Returns
#SEASONALITY: Monthly Seasonality
fig_ticker_Seasonality_Monthly = barChartPlotter.create_seasonality_fig(ticker_monthly_returns, f'Monthly Seasonality: {ticker_str}', 'monthly')
fig_ticker_Seasonality_Monthly.show()

#SEASONALITY: Weekly Seasonality
fig_ticker_Seasonality_weekly = barChartPlotter.create_seasonality_fig(ticker_weekly_returns, f'Weekly Seasonality: {ticker_str}', 'weekly')
fig_ticker_Seasonality_weekly.show()

#SEASONALITY: Daily Seasonality
fig_ticker_Seasonality_daily = barChartPlotter.create_seasonality_fig(ticker_daily_returns, f'Daily Seasonality: {ticker_str}', 'daily')
fig_ticker_Seasonality_daily.show()

In [ ]:
# Block 9: render interactive momentum z-score comparisons

window_pairs = {
    "21 vs 50": (21, 50),
    "50 vs 200": (50, 200),
    "200 vs 400": (200, 400),
}

zscore_data = momentum_analytics.momentum_zscore_map(
    ticker['Close'],
    window_pairs=window_pairs,
)

fig = lineChartPlotter.plot_momentum_zscore_comparison(
    zscore_data=zscore_data,
    ticker_label=ticker_str,
    default_label="50 vs 200",
    default_time_label="3 Years",
    sigma_levels=(0.5, 1.0, 1.5),
)
fig.show()


In [ ]:
# Block 10: compute Sharpe/Sortino ratios and spreads

asset_close = ticker['Close']

risk_context = risk_relative_analytics.build_sharpe_sortino_context(
    analytics=rolling,
    asset_close=asset_close,
    time_frame_map=time_frame_map,
    benchmark_data=benchmark_data,
)

asset_sharpe_map = risk_context['asset_sharpe_map']
asset_sortino_map = risk_context['asset_sortino_map']
asset_sharpe_sortino_spread_map = risk_context['asset_sharpe_sortino_spread_map']

benchmark_metrics = risk_context['benchmark_metrics']
benchmark_order = risk_context['benchmark_order']
default_benchmark = risk_context['default_benchmark']
spread_plot_data = risk_context['spread_plot_data']
term_config_map = risk_context['term_config_map']


In [ ]:
# Block 11: plot Sharpe & Sortino efficiency with dropdown controls

long_window = time_frame_map.get('long')
default_term_label = f"{long_window}-day" if long_window is not None else max(term_config_map, key=lambda label: int(str(label).split('-')[0]))

fig = lineChartPlotter.plot_sharpe_sortino_comparison(
    term_config_map=term_config_map,
    ticker_label=ticker_str,
    default_label=default_term_label,
)
fig.show()


In [ ]:
# Block 12: combine risk-adjusted return and benchmark plots

if benchmark_order:
    benchmark_plot_payload = risk_relative_analytics.build_benchmark_plot_payload(
        asset_sharpe_map=asset_sharpe_map,
        benchmark_metrics=benchmark_metrics,
        spread_plot_data=spread_plot_data,
        time_frame_map=time_frame_map,
    )

    default_term = 'long' if 'long' in time_frame_map else max(time_frame_map, key=time_frame_map.get)

    summary_fig = lineChartPlotter.plot_multi_benchmark_sharpe_spread_summary(
        summary_zscore_map=benchmark_plot_payload['summary_zscore_map'],
        time_frame_map=time_frame_map,
        ticker_label=ticker_str,
        default_term=default_term,
    )
    summary_fig.show()

    detail_fig = lineChartPlotter.plot_benchmark_zscore_detail(
        detail_zscore_map=benchmark_plot_payload['detail_zscore_map'],
        benchmark_order=benchmark_plot_payload['benchmark_order'],
        time_frame_map=time_frame_map,
        ticker_label=ticker_str,
        default_benchmark=benchmark_plot_payload['default_benchmark'],
        default_term=default_term,
    )
    detail_fig.show()
else:
    print('No benchmark data available for benchmark comparison plots.')


In [ ]:
# Block 13: analyze drawdown, skew, kurtosis, and Gini z-scores

dd_window_options = [21, 50, 200]

risk_distribution_context = risk_distribution_analytics.build_risk_distribution_context(
    close_series=ticker['Close'],
    windows=dd_window_options,
    default_window=200 if 200 in dd_window_options else max(dd_window_options),
)

fig = lineChartPlotter.plot_risk_distribution_zscores(
    metrics_by_window=risk_distribution_context['metrics_by_window'],
    window_options=risk_distribution_context['windows'],
    default_window=risk_distribution_context['default_window'],
    ticker_label=ticker_str,
)
fig.show()


In [ ]:
#Idiosyncratic risk via Fama-French Factor Analysis
analysis_period = "max"
interval = "1d"
rolling_window = 252

ff_proxy = model.run_ff5_proxy_analysis(
    asset_ticker=ticker_str,
    period=analysis_period,
    interval=interval,
    window=rolling_window,
    auto_window=True,
    verbose=True,
)

prices_df = ff_proxy["proxy_prices"]
returns = ff_proxy["proxy_returns"]
factor_returns = ff_proxy["factor_returns_all"]
factor_returns_capm = ff_proxy["factor_returns_capm"]
factor_returns_ff3 = ff_proxy["factor_returns_ff3"]
factor_returns_ff5 = ff_proxy["factor_returns_ff5"]
factor_returns_ff5_custom = factor_returns_ff5.copy()
stock_returns = ff_proxy["stock_returns"]
rolling_results_ff5_custom = ff_proxy["rolling_results"]

#Plotting the Fama-French Factor Analysis Results
qp.plot_rolling_regression(rolling_results_ff5_custom, ticker_str, factor_returns_ff5_custom)
qp.plot_idiosyncratic_risk(rolling_results_ff5_custom, ticker_str)
